# Module 3.6: Episodic Memory Lifecycle

An agent stores episodic events (past trips, feedback, preferences) over months of
interaction. Without lifecycle management, two problems emerge:

1. **Stale events pollute retrieval** — A hotel rating from 2 years ago (since renovated)
   leads to wrong recommendations
2. **One-session signals don't generalize** — Sarah ordered vegetarian once; is she
   vegetarian, or was it just that day?

> **The question**: How do we expire old events and detect which patterns are durable
> enough to become permanent semantic memory?

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys, os, json
import sniffio
from datetime import datetime, timezone, timedelta
from collections import defaultdict

sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")

from lifecycle_utils import (
    EpisodicTTLConfig, EpisodicPattern, MemoryItem, MemoryState
)
from shared.travel_agent import create_client

client, credential = create_client("../.env")
print("Setup complete")

## The Problem: Stale Events and Premature Generalisation

Without lifecycle management, episodic memory has two failure modes.
Let's demonstrate both:

In [ ]:
# Failure Mode 1: STALE EVENTS
# Sarah rated a hotel 2 years ago — but it's been renovated since
stale_events = [
    {"event": "Rated Marriott Midtown 2/5 — noisy room, bad AC", "date": "2024-01-15",
     "age_days": 540, "still_relevant": False},  # Hotel renovated in 2024!
    {"event": "Rated Marriott Midtown 5/5 — great after renovation", "date": "2025-06-20",
     "age_days": 19, "still_relevant": True},
]

print("=== Failure Mode 1: Stale Events ===\n")
print("Query: 'How was Sarah's experience at Marriott Midtown?'\n")
print("Without TTL, BOTH events are retrieved:")
for e in stale_events:
    icon = "✅" if e["still_relevant"] else "❌"
    print(f"  {icon} [{e['date']}] {e['event']} (age: {e['age_days']}d)")
print("\n→ The 2024 review is MISLEADING — the hotel has been renovated since")
print("→ Agent might avoid recommending it based on outdated feedback\n")

print("=" * 60)

# Failure Mode 2: PREMATURE GENERALISATION
# Sarah ordered vegetarian ONCE — is she vegetarian?
single_session_events = [
    {"event": "Ordered vegetarian meal", "session": "session-A", "count": 1},
]

cross_session_events = [
    {"event": "Ordered vegetarian meal", "session": "session-A", "count": 1},
    {"event": "Asked for vegetarian options", "session": "session-B", "count": 1},
    {"event": "Confirmed vegetarian diet", "session": "session-C", "count": 1},
    {"event": "Requested vegetarian airline meal", "session": "session-D", "count": 1},
]

print("\n=== Failure Mode 2: Premature Generalisation ===\n")
print("Single session (1 event, 1 session):")
print(f"  Observed: 'Ordered vegetarian meal' in session-A only")
print(f"  Naive conclusion: 'Sarah is vegetarian' — ❌ Maybe she just wasn't hungry")
print(f"\nCross-session (4 events, 4 sessions):")
for e in cross_session_events:
    print(f"  • {e['event']} ({e['session']})")
print(f"  Confident conclusion: 'Sarah is vegetarian' — ✅ Pattern is durable")
print(f"\n→ Without cross-session thresholds, one-off events get promoted incorrectly")

## The Solution: TTL-Based Expiry + Cross-Session Graduation

Two mechanisms solve these problems:

| Mechanism | Solves | How |
|-----------|--------|-----|
| **TTL (Time-to-Live)** | Stale events | Events auto-expire after configurable days (Cosmos DB native) |
| **Cross-session graduation** | Premature generalisation | Patterns must appear across 3+ distinct sessions before becoming semantic memory |

### Tiered TTL Strategy

| Event Type | TTL | Rationale |
|------------|-----|------------|
| `trip` | 365 days | Historical reference, useful for "same as last time" |
| `feedback` | 180 days | Preferences evolve; old feedback may be stale |
| `preference` | 90 days | Unless confirmed across sessions → graduates to semantic |

## Building the TTL Configuration

Cosmos DB supports document-level TTL natively. When a document has a `ttl` field
(value in seconds), Cosmos automatically deletes it after that duration.

**Container setup** (TTL must be enabled at container level):
```python
container = database.create_container_if_not_exists(
    id="episodic-events",
    partition_key=PartitionKey(path="/user_id"),
    default_ttl=-1,  # Enable TTL but no default (per-document)
)
```

In [ ]:
ttl_config = EpisodicTTLConfig(
    default_ttl_days=180,
    ttl_by_event_type={
        "trip": 365,
        "feedback": 180,
        "preference": 90,
    },
    graduation_threshold=3,     # Need 3+ occurrences
    graduation_session_min=3,   # Across 3+ distinct sessions
)

print("TTL Configuration:")
for etype, days in ttl_config.ttl_by_event_type.items():
    secs = ttl_config.compute_ttl_seconds(etype)
    print(f"  {etype:<12} → {days} days ({secs:,} seconds)")

print(f"\nGraduation requires: {ttl_config.graduation_threshold}+ events "
      f"across {ttl_config.graduation_session_min}+ sessions")

## Adding TTL to Episodic Events

When storing an episodic event, we compute the TTL based on event type and
add it to the document before writing to Cosmos.

In [ ]:
import uuid

def create_episodic_event(
    user_id: str,
    event_type: str,
    description: str,
    details: dict = None,
    session_id: str = None,
    timestamp: datetime = None,
) -> dict:
    """Create an episodic event document with TTL."""
    timestamp = timestamp or datetime.now(timezone.utc)
    ttl_seconds = ttl_config.compute_ttl_seconds(event_type)

    return {
        "id": f"{user_id}-{uuid.uuid4().hex[:8]}",
        "user_id": user_id,
        "event_type": event_type,
        "description": description,
        "details": details or {},
        "session_id": session_id or str(uuid.uuid4()),
        "timestamp": timestamp.isoformat(),
        "ttl": ttl_seconds,                # Cosmos DB auto-deletes after this
        "graduated": False,                 # Set True when promoted to semantic
    }


# Demo: create events with different TTLs
sample_events = [
    create_episodic_event("E001", "trip", "Traveled to NYC for engineering summit",
                          {"hotel": "Marriott Midtown", "rating": 5}),
    create_episodic_event("E001", "feedback", "Rated Marriott Midtown 5/5",
                          {"hotel_chain": "Marriott", "rating": 5}),
    create_episodic_event("E001", "preference", "Requested aisle seat",
                          {"seat_type": "aisle"}),
]

print("Sample events with TTL:")
for event in sample_events:
    days = event["ttl"] // 86400
    print(f"  [{event['event_type']:<10}] {event['description'][:40]} — TTL: {days} days")

## Cross-Session Graduation

The key lifecycle transition for episodic memory: when a pattern appears
consistently across **multiple independent sessions**, it graduates to semantic memory.

### Why Cross-Session?

Within a single session, a user might mention "I like Marriott" three times —
but that's one conversation, not proof of a durable preference. Requiring the
pattern across 3+ distinct sessions ensures it's a genuine, persistent preference.

### How It Works

```
Session A: Sarah rates Marriott 5/5 after NYC trip
Session B: Sarah asks to book Marriott again in Chicago
Session C: Sarah says "I always stay at Marriott"
                ↓
Pattern detected: hotel_preference = "Marriott" (3 events, 3 sessions)
                ↓
Graduates to semantic: MemoryItem("Prefers Marriott hotels", state=PROVISIONAL)
```

In [ ]:
class GraduationEngine:
    """Detects cross-session patterns in episodic events and graduates them to semantic."""

    # Pattern extraction rules: (event detail key → pattern type)
    PATTERN_EXTRACTORS = {
        "hotel_chain": "hotel_preference",
        "airline": "airline_preference",
        "seat_type": "seat_preference",
        "dietary": "dietary_preference",
        "flight_class": "class_preference",
    }

    def __init__(self, config: EpisodicTTLConfig = None):
        self.config = config or EpisodicTTLConfig()

    def detect_patterns(self, events: list[dict]) -> list[EpisodicPattern]:
        """Scan events and find repeating patterns across sessions."""
        # Group by (pattern_type, value)
        pattern_map: dict[tuple[str, str], EpisodicPattern] = {}

        for event in events:
            if event.get("graduated"):
                continue  # Skip already-graduated events

            details = event.get("details", {})
            session_id = event.get("session_id", "unknown")

            for detail_key, pattern_type in self.PATTERN_EXTRACTORS.items():
                if detail_key in details and details[detail_key]:
                    value = str(details[detail_key])
                    key = (pattern_type, value)

                    if key not in pattern_map:
                        pattern_map[key] = EpisodicPattern(
                            pattern_type=pattern_type,
                            value=value,
                        )
                    p = pattern_map[key]
                    p.event_count += 1
                    p.session_ids.append(session_id)
                    p.event_ids.append(event["id"])

            # Also check high ratings as implicit preference
            if details.get("rating", 0) >= 4 and "hotel_chain" in details:
                # High rating = implicit hotel preference signal
                pass  # Already captured above via hotel_chain key

        # Filter to qualifying patterns
        return [p for p in pattern_map.values() if p.qualifies(self.config)]

    def graduate(self, pattern: EpisodicPattern, user_id: str) -> MemoryItem:
        """Convert a qualifying pattern into a semantic MemoryItem."""
        content = self._generate_content(pattern)
        return MemoryItem(
            user_id=user_id,
            content=content,
            category="preference",
            state=MemoryState.PROVISIONAL,  # NOT trusted yet!
            confidence=min(0.5 + (pattern.session_count * 0.1), 0.8),
            confirmation_count=pattern.event_count,
            source_type="llm_inference",  # Inferred from events, not user-stated
        )

    def _generate_content(self, pattern: EpisodicPattern) -> str:
        """Generate human-readable content for the graduated memory."""
        templates = {
            "hotel_preference": f"Prefers {pattern.value} hotels",
            "airline_preference": f"Prefers {pattern.value} airline",
            "seat_preference": f"Prefers {pattern.value} seat",
            "dietary_preference": f"Dietary preference: {pattern.value}",
            "class_preference": f"Prefers {pattern.value} class flights",
        }
        return templates.get(pattern.pattern_type,
                             f"Recurring preference: {pattern.value}")


graduation_engine = GraduationEngine(ttl_config)
print("GraduationEngine ready")

## The Payoff: Cross-Session Pattern Detection

Let's simulate Sarah's events across 5 different sessions over 3 months.
Watch which patterns qualify for graduation — and which one-off events correctly
get filtered out.

In [ ]:
# Simulate events across 5 sessions
SESSION_A = "session-2026-01-15"
SESSION_B = "session-2026-02-03"
SESSION_C = "session-2026-02-20"
SESSION_D = "session-2026-03-10"
SESSION_E = "session-2026-04-01"

now = datetime.now(timezone.utc)
simulated_events = [
    # Session A: NYC trip, stayed at Marriott, rated 5/5
    create_episodic_event("E001", "trip", "NYC trip for engineering summit",
                          {"hotel_chain": "Marriott", "rating": 5, "airline": "United"},
                          session_id=SESSION_A,
                          timestamp=now - timedelta(days=170)),
    create_episodic_event("E001", "feedback", "Loved the Marriott Midtown",
                          {"hotel_chain": "Marriott", "rating": 5},
                          session_id=SESSION_A,
                          timestamp=now - timedelta(days=170)),

    # Session B: Booked Chicago, asked for Marriott
    create_episodic_event("E001", "preference", "Requested Marriott for Chicago trip",
                          {"hotel_chain": "Marriott"},
                          session_id=SESSION_B,
                          timestamp=now - timedelta(days=152)),
    create_episodic_event("E001", "preference", "Requested aisle seat",
                          {"seat_type": "aisle"},
                          session_id=SESSION_B,
                          timestamp=now - timedelta(days=152)),

    # Session C: Another trip, Marriott again, aisle seat
    create_episodic_event("E001", "trip", "SF team offsite",
                          {"hotel_chain": "Marriott", "rating": 4, "airline": "United",
                           "seat_type": "aisle"},
                          session_id=SESSION_C,
                          timestamp=now - timedelta(days=135)),

    # Session D: Booked London, asked for United and Marriott
    create_episodic_event("E001", "preference", "Wants United for London flight",
                          {"airline": "United"},
                          session_id=SESSION_D,
                          timestamp=now - timedelta(days=117)),
    create_episodic_event("E001", "preference", "Requested aisle seat for long flight",
                          {"seat_type": "aisle"},
                          session_id=SESSION_D,
                          timestamp=now - timedelta(days=117)),

    # Session E: Yet another Marriott + United booking
    create_episodic_event("E001", "trip", "Tokyo conference",
                          {"hotel_chain": "Marriott", "airline": "United",
                           "seat_type": "aisle", "rating": 5},
                          session_id=SESSION_E,
                          timestamp=now - timedelta(days=96)),

    # Noise: one-off events that should NOT graduate
    create_episodic_event("E001", "feedback", "Tried the Hilton, it was okay",
                          {"hotel_chain": "Hilton", "rating": 3},
                          session_id=SESSION_C,
                          timestamp=now - timedelta(days=135)),
    create_episodic_event("E001", "preference", "Window seat for short flight",
                          {"seat_type": "window"},
                          session_id=SESSION_A,
                          timestamp=now - timedelta(days=170)),
]

print(f"Simulated {len(simulated_events)} events across 5 sessions")
print(f"\nEvents by session:")
by_session = defaultdict(list)
for e in simulated_events:
    by_session[e['session_id']].append(e)
for sid, events in sorted(by_session.items()):
    print(f"  {sid}: {len(events)} events")

In [ ]:
# Detect patterns
patterns = graduation_engine.detect_patterns(simulated_events)

print(f"=== Detected Patterns ({len(patterns)} qualifying) ===")
print()
for p in patterns:
    print(f"  ✅ {p.pattern_type}: {p.value}")
    print(f"     Events: {p.event_count} | Sessions: {p.session_count} "
          f"(need {ttl_config.graduation_threshold}+ events, "
          f"{ttl_config.graduation_session_min}+ sessions)")
    print()

In [ ]:
# What did NOT qualify? Let's check all patterns including non-qualifying ones
all_patterns = {}
for event in simulated_events:
    details = event.get("details", {})
    for detail_key, pattern_type in graduation_engine.PATTERN_EXTRACTORS.items():
        if detail_key in details and details[detail_key]:
            key = (pattern_type, str(details[detail_key]))
            if key not in all_patterns:
                all_patterns[key] = {"count": 0, "sessions": set()}
            all_patterns[key]["count"] += 1
            all_patterns[key]["sessions"].add(event["session_id"])

print("=== All Detected Patterns (including non-qualifying) ===")
print()
for (ptype, value), info in sorted(all_patterns.items()):
    qualifies = (info["count"] >= ttl_config.graduation_threshold and
                 len(info["sessions"]) >= ttl_config.graduation_session_min)
    icon = "✅" if qualifies else "❌"
    print(f"  {icon} {ptype:<20} = {value:<10} | events={info['count']} sessions={len(info['sessions'])}")

print(f"\n→ 'Hilton' and 'window' don't qualify: too few sessions/events")

## Graduating to Semantic Memory

Qualifying patterns get promoted to `MemoryItem` objects that enter the
semantic memory lifecycle (Notebooks 03–05). Note they start as **PROVISIONAL** —
cross-session consistency is strong evidence, but not as authoritative as a
direct user statement.

In [ ]:
# Graduate qualifying patterns
graduated_memories = []
for pattern in patterns:
    memory = graduation_engine.graduate(pattern, user_id="E001")
    graduated_memories.append(memory)

print(f"=== Graduated Memories ({len(graduated_memories)}) ===")
print()
for m in graduated_memories:
    print(f"  [{m.state.value:<12}] {m.content}")
    print(f"               confidence={m.confidence:.2f} | "
          f"confirmations={m.confirmation_count} | source={m.source_type}")
    print()

print("→ These enter the semantic memory lifecycle:")
print("  • PROVISIONAL state — agent uses them with hedging language")
print("  • Need user confirmation to reach TRUSTED")
print("  • Subject to belief revision if user changes preference")

## Marking Events as Graduated

After graduation, source events get marked `graduated: True` so they won't
trigger re-graduation on the next scan. The events still exist until their
TTL expires — graduation doesn't delete them.

In [ ]:
def mark_graduated(events: list[dict], patterns: list[EpisodicPattern]) -> int:
    """Mark source events as graduated. Returns count of marked events."""
    graduated_ids = set()
    for p in patterns:
        graduated_ids.update(p.event_ids)

    count = 0
    for event in events:
        if event["id"] in graduated_ids and not event.get("graduated"):
            event["graduated"] = True
            count += 1
    return count


marked = mark_graduated(simulated_events, patterns)
print(f"Marked {marked} events as graduated")
print(f"\nEvent status:")
for e in simulated_events:
    status = "GRADUATED" if e.get("graduated") else "active"
    days_remaining = e["ttl"] // 86400  # Simplified; real TTL counts from creation
    print(f"  [{status:<9}] {e['description'][:45]:<45} TTL: {days_remaining}d")

## Production: Cosmos DB Integration

In production, the graduation engine runs as a periodic batch job:

```python
# Query all non-graduated events for a user
query = "SELECT * FROM c WHERE c.user_id = @uid AND (c.graduated = false OR NOT IS_DEFINED(c.graduated))"

# Detect patterns
patterns = graduation_engine.detect_patterns(events)

# For each qualifying pattern:
#   1. Create MemoryItem in semantic memory container
#   2. Update source events with graduated=true
#   3. (Optional) Extend TTL on graduated events for audit trail
```

The TTL is set at document creation time. Cosmos handles deletion automatically —
no sweep job needed.

In [ ]:
# Show what a Cosmos upsert looks like with TTL
sample = simulated_events[0]
print("=== Cosmos Document (with TTL) ===")
print(json.dumps(sample, indent=2, default=str))
print(f"\n→ Cosmos will auto-delete this document {sample['ttl']//86400} days after creation")
print(f"→ No application-level sweep needed")

## The Full Episodic Lifecycle

```mermaid
flowchart TD
    A[Event occurs] --> B[Store with TTL + session_id]
    B --> C{TTL expires?}
    C -->|Yes| D[Cosmos auto-deletes]
    C -->|No| E{Pattern across 3+ sessions?}
    E -->|No| C
    E -->|Yes| F[Graduate to semantic memory\nstate=PROVISIONAL]
    F --> G[Mark source events graduated=true]
    G --> C
```

## Key Takeaways

1. **TTL is native** — Cosmos DB handles expiry, no application sweep needed
2. **Tiered TTL** — different event types have different retention needs
3. **Cross-session is key** — requiring 3+ distinct sessions filters noise from signal
4. **Graduation creates PROVISIONAL** — not TRUSTED (still needs user confirmation)
5. **Events aren't deleted on graduation** — they expire naturally via TTL
6. **session_id is required** — without it, cross-session detection is impossible

## Next: Procedural Lifecycle (Notebook 07)

Episodic events have clear expiry criteria (time-based). Procedural memory is harder:
how do you know if a learned procedure is still valid when the policies it was based
on might have changed? The next notebook uses RAG as ground truth to validate
stored procedures.